<a href="https://colab.research.google.com/github/group6aiml-hub/NIDS_AInML/blob/notebooks/notebooks/07_BaselineComparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



**This is a baseline comparison that the models will be compared against**



In [ ]:
!pip install -q imbalanced-learn xgboost

from google.colab import drive
drive.mount('/content/drive')

import time
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer, LabelEncoder
from sklearn.metrics import f1_score, recall_score, classification_report
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

path = '/content/drive/MyDrive/Datasets/'
train = pd.read_csv(path + 'train_clean.csv')
test  = pd.read_csv(path + 'test_clean.csv')

X_train = train.drop(columns=['label', 'attack_cat'])
X_tr, X_val = train_test_split(X_train, test_size=0.2,
                               stratify=train['attack_cat'], random_state=42)

# Fasit for begge oppgaver, med samme split
y_tr_bin, y_val_bin = train.loc[X_tr.index, 'label'], train.loc[X_val.index, 'label']
le = LabelEncoder().fit(train['attack_cat'])
y_tr_mc  = le.transform(train.loc[X_tr.index,  'attack_cat'])
y_val_mc = le.transform(train.loc[X_val.index, 'attack_cat'])

print(X_tr.shape, X_val.shape)

Mounted at /content/drive
(86192, 41) (21548, 41)


In [ ]:
#Preprocess og felles test funksjon

cat_cols   = ['proto', 'service', 'state']
num_cols   = X_tr.columns.drop(cat_cols).tolist()
skew       = X_tr[num_cols].skew()
skew_cols  = skew[skew.abs() > 1].index.tolist()
other_cols = [c for c in num_cols if c not in skew_cols]

def lag_preprocess():
    return ColumnTransformer([
        ('cat',  OneHotEncoder(handle_unknown='ignore'), cat_cols),
        ('skew', Pipeline([('log', FunctionTransformer(np.log1p)),
                           ('scale', StandardScaler())]), skew_cols),
        ('num',  StandardScaler(), other_cols),
    ])

# SMOTE med samme strategi som i 05/06, men med tallkoder i stedet for navn
smote_strategi = {le.transform([k])[0]: v for k, v in
                  {'Worms': 3000, 'Shellcode': 3000, 'Backdoor': 3000,
                   'Analysis': 3000, 'DoS': 5000}.items()}

resultater = []

def test_modell(navn, lag_clf):
    """lag_clf er en funksjon som lager en NY modell hver gang den kalles."""
    rad = {'Modell': navn}

    # Binær (uten SMOTE)
    p = Pipeline([('prep', lag_preprocess()), ('clf', lag_clf())])
    t = time.time(); p.fit(X_tr, y_tr_bin); rad['Tid binær (s)'] = round(time.time() - t, 1)
    pred = p.predict(X_val)
    rad['Binær macro-F1']  = f1_score(y_val_bin, pred, average='macro')
    rad['Recall angrep']   = recall_score(y_val_bin, pred)

    # Multiklasse (med SMOTE)
    p = ImbPipeline([('prep', lag_preprocess()),
                     ('smote', SMOTE(sampling_strategy=smote_strategi, random_state=42)),
                     ('clf', lag_clf())])
    t = time.time(); p.fit(X_tr, y_tr_mc); rad['Tid multi (s)'] = round(time.time() - t, 1)
    pred = p.predict(X_val)
    rad['Multi macro-F1'] = f1_score(y_val_mc, pred, average='macro')

    print(f'===== {navn} (multiklasse)')
    print(classification_report(y_val_mc, pred, target_names=le.classes_, digits=3, zero_division=0))
    resultater.append(rad)

In [ ]:
#Test med modellene vi allerede har

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

test_modell('Logistic Regression', lambda: LogisticRegression(max_iter=1000))
test_modell('Random Forest',       lambda: RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))

pd.DataFrame(resultater).set_index('Modell').round(3)

===== Logistic Regression (multiklasse)
                precision    recall  f1-score   support

      Analysis      0.173     0.354     0.232       319
      Backdoor      0.180     0.394     0.247       307
           DoS      0.222     0.057     0.090       761
      Exploits      0.773     0.730     0.751      3969
       Fuzzers      0.605     0.669     0.635      3230
       Generic      0.950     0.794     0.865       836
        Normal      0.937     0.876     0.906     10378
Reconnaissance      0.602     0.521     0.559      1504
     Shellcode      0.277     0.748     0.404       218
         Worms      0.034     0.538     0.063        26

      accuracy                          0.745     21548
     macro avg      0.475     0.568     0.475     21548
  weighted avg      0.779     0.745     0.757     21548

===== Random Forest (multiklasse)
                precision    recall  f1-score   support

      Analysis      0.085     0.103     0.093       319
      Backdoor      0.167 

,Tid binær (s),Binær macro-F1,Recall angrep,Tid multi (s),Multi macro-F1
Modell,,,,,
Logistic Regression,10.8,0.908,0.956,79.1,0.475
Random Forest,23.9,0.932,0.951,35.3,0.572


In [ ]:
# XGBoost

from xgboost import XGBClassifier

test_modell('XGBoost', lambda: XGBClassifier(n_estimators=300, learning_rate=0.1, max_depth=8,
                                             tree_method='hist', n_jobs=-1, random_state=42))

pd.DataFrame(resultater).set_index('Modell').round(3)

===== XGBoost (multiklasse)
                precision    recall  f1-score   support

      Analysis      0.080     0.154     0.105       319
      Backdoor      0.126     0.238     0.165       307
           DoS      0.328     0.254     0.286       761
      Exploits      0.839     0.853     0.846      3969
       Fuzzers      0.758     0.770     0.764      3230
       Generic      0.966     0.841     0.899       836
        Normal      0.948     0.924     0.936     10378
Reconnaissance      0.871     0.733     0.796      1504
     Shellcode      0.577     0.839     0.684       218
         Worms      0.515     0.654     0.576        26

      accuracy                          0.825     21548
     macro avg      0.601     0.626     0.606     21548
  weighted avg      0.844     0.825     0.833     21548



,Tid binær (s),Binær macro-F1,Recall angrep,Tid multi (s),Multi macro-F1
Modell,,,,,
Logistic Regression,10.8,0.908,0.956,79.1,0.475
Random Forest,23.9,0.932,0.951,35.3,0.572
XGBoost,15.1,0.938,0.956,154.0,0.606


Binær: 0.938, mot 0.932 for Random Forest. Liten forbedring, med samme recall for angrep (0.956).

Multiklasse: 0.606, mot 0.572. Det er den største forbedringen vi har fått så langt, større enn alle ubalansetiltakene i 05. SMOTE ga +0.021, mens bytte av modell gir +0.034.

Ulempe: multiklasse tar 154 s, over fire ganger så lenge som Random Forest. XGBoost bygger egne trær for hver av de 10 klassene i hver runde.